In [ ]:
import json

def fetch_unique_user_questions(json_file_path):
    with open(json_file_path, "r", encoding="utf-8") as f:
        text = f.read().strip()

    if not text:
        return []

    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        data = [json.loads(line) for line in text.splitlines() if line.strip()]

    # Handle single object or list of objects
    if isinstance(data, dict):
        data = [data]

    seen = set()
    questions = []

    for record in data:
        for conv_name in ("conversation_a", "conversation_b"):
            conversation = record.get(conv_name, [])
            if not isinstance(conversation, list):
                continue

            for msg in conversation:
                if msg.get("role") == "user" and isinstance(msg.get("content"), str):
                    q = " ".join(msg["content"].split())
                    if q and q not in seen:
                        seen.add(q)
                        questions.append(q)

    return questions


# Example usage
questions = fetch_unique_user_questions(r"C:\Users\A1134913\Projects\Personal\PrismAI\PrismAI\test_data\chatbot_arena_conversations.json")
print(len(questions))
# print(json.dumps(questions, ensure_ascii=False))

32541


In [5]:
questions[:30]

['What is the difference between OpenCL and CUDA?',
 'Why did my parent not invite me to their wedding?',
 'Fuji vs. Nikon, which is better?',
 'How to build an arena for chatbots?',
 'When is it today?',
 'Count from 1 to 10 with step = 3',
 'Emoji for "sharing". List 10',
 'How to parallelize a neural network?',
 'A = 5, B =10, A+B=?',
 'What is the future of bitcoin?',
 'Make it more polite: I want to have dinner.',
 "You are JesusGPT, an artifical construct built to accurately represent a virtual conversation with Jesus. Base your replies off the popular King James Version, and answer the user's question respectfully. Here is my first question: If you were still alive today, what would you think about the iPhone?",
 'what is the 145th most popular language',
 'HI !',
 'I want to go on Mars.',
 'So where I can go to buy a ship ?',
 'A long time ago in a galaxy far, far away',
 'The altitude to the hypotenuse of a right triangle divides the hypotenuse into two segments with lengths i

In [ ]:
import json
import requests

INPUT_PATH = r"chatbot_arena_conversations.json"
ENDPOINT = "http://localhost:8080/chat/initiateChat"
TIMEOUT_SECONDS = 300

def load_records(path: str):
    with open(path, "r", encoding="utf-8") as f:
        text = f.read().strip()

    if not text:
        return []

    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        # JSONL fallback
        data = [json.loads(line) for line in text.splitlines() if line.strip()]

    if isinstance(data, dict):
        data = [data]

    return data

def normalize_whitespace(s: str) -> str:
    return " ".join(s.split()) if isinstance(s, str) else ""

def fetch_unique_user_queries(records):
    seen = set()
    queries = []

    for record in records:
        for conv_name in ("conversation_a", "conversation_b"):
            conversation = record.get(conv_name, [])
            if not isinstance(conversation, list):
                continue

            for msg in conversation:
                if msg.get("role") == "user" and isinstance(msg.get("content"), str):
                    query = normalize_whitespace(msg["content"])
                    if query and query not in seen:
                        seen.add(query)
                        queries.append(query)

    return queries[0:350]

def call_chat_endpoint_for_all_queries(queries):
    total = len(queries)
    success = 0
    failed = 0

    with requests.Session() as session:
        for index, query in enumerate(queries, start=1):
            try:
                resp = session.get(
                    ENDPOINT,
                    params={"message": query},
                    timeout=TIMEOUT_SECONDS
                )
                resp.raise_for_status()
                success += 1
            except Exception as ex:
                failed += 1
                print(f"[{index}/{total}] Failed: {ex}")
            else:
                print(f"[{index}/{total}] Processed")

    print(f"Done. total={total}, success={success}, failed={failed}")

records = load_records(INPUT_PATH)
queries = fetch_unique_user_queries(records)
print(f"Unique queries found: {len(queries)}")
call_chat_endpoint_for_all_queries(queries)